In [0]:
%sql
-- Import Chinook tables (Idempotent)
-- NOTE (M7): chinook_playlist and chinook_mediatype are loaded but not used downstream
-- Consider removing these to reduce compute/egress costs 
CREATE OR REPLACE TABLE chinook_artist AS
SELECT * FROM read_files(
  'r2://ftw-b12-dataengineering@6338489909d41c2f78a0a2345a684267.r2.cloudflarestorage.com/shared/week05/chinook_csv/Artist.csv',
  format => 'csv', header => true,
  schema => 'ArtistId INT, Name STRING'
);

CREATE OR REPLACE TABLE chinook_album AS
SELECT * FROM read_files(
  'r2://ftw-b12-dataengineering@6338489909d41c2f78a0a2345a684267.r2.cloudflarestorage.com/shared/week05/chinook_csv/Album.csv',
  format => 'csv', header => true,
  schema => 'AlbumId INT, Title STRING, ArtistId INT'
);

CREATE OR REPLACE TABLE chinook_genre AS
SELECT * FROM read_files(
  'r2://ftw-b12-dataengineering@6338489909d41c2f78a0a2345a684267.r2.cloudflarestorage.com/shared/week05/chinook_csv/Genre.csv',
  format => 'csv', header => true,
  schema => 'GenreId INT, Name STRING'
);

CREATE OR REPLACE TABLE chinook_mediatype AS
SELECT * FROM read_files(
  'r2://ftw-b12-dataengineering@6338489909d41c2f78a0a2345a684267.r2.cloudflarestorage.com/shared/week05/chinook_csv/MediaType.csv',
  format => 'csv', header => true,
  schema => 'MediaTypeId INT, Name STRING'
);

CREATE OR REPLACE TABLE chinook_track AS
SELECT * FROM read_files(
  'r2://ftw-b12-dataengineering@6338489909d41c2f78a0a2345a684267.r2.cloudflarestorage.com/shared/week05/chinook_csv/Track.csv',
  format => 'csv', header => true,
  schema => 'TrackId INT, Name STRING, AlbumId INT, MediaTypeId INT, GenreId INT, Composer STRING, Milliseconds INT, Bytes INT, UnitPrice DECIMAL(10,2)'
);

CREATE OR REPLACE TABLE chinook_playlist AS
SELECT * FROM read_files(
  'r2://ftw-b12-dataengineering@6338489909d41c2f78a0a2345a684267.r2.cloudflarestorage.com/shared/week05/chinook_csv/Playlist.csv',
  format => 'csv', header => true,
  schema => 'PlaylistId INT, Name STRING'
);

CREATE OR REPLACE TABLE chinook_playlisttrack AS
SELECT * FROM read_files(
  'r2://ftw-b12-dataengineering@6338489909d41c2f78a0a2345a684267.r2.cloudflarestorage.com/shared/week05/chinook_csv/PlaylistTrack.csv',
  format => 'csv', header => true,
  schema => 'PlaylistId INT, TrackId INT'
);

CREATE OR REPLACE TABLE chinook_employee AS
SELECT * FROM read_files(
  'r2://ftw-b12-dataengineering@6338489909d41c2f78a0a2345a684267.r2.cloudflarestorage.com/shared/week05/chinook_csv/Employee.csv',
  format => 'csv', header => true,
  schema => 'EmployeeId INT, LastName STRING, FirstName STRING, Title STRING, ReportsTo INT, BirthDate TIMESTAMP, HireDate TIMESTAMP, Address STRING, City STRING, State STRING, Country STRING, PostalCode STRING, Phone STRING, Fax STRING, Email STRING'
);

CREATE OR REPLACE TABLE chinook_customer AS
SELECT * FROM read_files(
  'r2://ftw-b12-dataengineering@6338489909d41c2f78a0a2345a684267.r2.cloudflarestorage.com/shared/week05/chinook_csv/Customer.csv',
  format => 'csv', header => true,
  schema => 'CustomerId INT, FirstName STRING, LastName STRING, Company STRING, Address STRING, City STRING, State STRING, Country STRING, PostalCode STRING, Phone STRING, Fax STRING, Email STRING, SupportRepId INT'
);

CREATE OR REPLACE TABLE chinook_invoice AS
SELECT * FROM read_files(
  'r2://ftw-b12-dataengineering@6338489909d41c2f78a0a2345a684267.r2.cloudflarestorage.com/shared/week05/chinook_csv/Invoice.csv',
  format => 'csv', header => true,
  schema => 'InvoiceId INT, CustomerId INT, InvoiceDate TIMESTAMP, BillingAddress STRING, BillingCity STRING, BillingState STRING, BillingCountry STRING, BillingPostalCode STRING, Total DECIMAL(10,2)'
);

CREATE OR REPLACE TABLE chinook_invoiceline AS
SELECT * FROM read_files(
  'r2://ftw-b12-dataengineering@6338489909d41c2f78a0a2345a684267.r2.cloudflarestorage.com/shared/week05/chinook_csv/InvoiceLine.csv',
  format => 'csv', header => true,
  schema => 'InvoiceLineId INT, InvoiceId INT, TrackId INT, UnitPrice DECIMAL(10,2), Quantity INT'
);

In [0]:
%sql
CREATE OR REPLACE TABLE silver_genre AS
SELECT
    GenreId,
    TRIM(Name) AS GenreName
FROM workspace.default.chinook_genre
WHERE GenreId IS NOT NULL
  AND Name IS NOT NULL;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE silver_artist AS
SELECT
    ArtistId,
    TRIM(Name) AS ArtistName
FROM workspace.default.chinook_artist
WHERE ArtistId IS NOT NULL
  AND Name IS NOT NULL;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE silver_album AS
SELECT
    a.AlbumId,
    TRIM(a.Title) AS AlbumTitle,
    a.ArtistId,
    COALESCE(TRIM(ar.ArtistName), 'Unknown Artist') AS ArtistName
FROM workspace.default.chinook_album a
LEFT JOIN silver_artist ar
    ON a.ArtistId = ar.ArtistId
WHERE a.AlbumId IS NOT NULL
  AND a.Title IS NOT NULL;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE silver_track AS
SELECT
    t.TrackId,
    TRIM(t.Name) AS TrackName,
    t.AlbumId,
    COALESCE(al.AlbumTitle, 'Unknown Album') AS Title,
    COALESCE(al.ArtistId, -1) AS ArtistId,
    COALESCE(al.ArtistName, 'Unknown Artist') AS ArtistName,
    t.GenreId,
    COALESCE(g.GenreName, 'Unknown Genre') AS GenreName,
    t.MediaTypeId,
    TRIM(t.Composer) AS Composer,
    t.Milliseconds,
    ROUND(t.Milliseconds / 60000.0, 2) AS DurationMinutes,
    t.Bytes,
    ROUND(t.Bytes / 1048576.0, 4) AS FileSizeMb,
    t.UnitPrice
FROM workspace.default.chinook_track t
LEFT JOIN silver_album al
    ON t.AlbumId = al.AlbumId
LEFT JOIN silver_genre g
    ON t.GenreId = g.GenreId
WHERE t.TrackId IS NOT NULL
  AND t.Name IS NOT NULL
  AND t.MediaTypeId != 3;
  
  SELECT * FROM silver_track LIMIT 10

TrackId,TrackName,AlbumId,Title,ArtistId,ArtistName,GenreId,GenreName,MediaTypeId,Composer,Milliseconds,DurationMinutes,Bytes,FileSizeMb,UnitPrice
1,For Those About To Rock (We Salute You),1,For Those About To Rock We Salute You,1,AC/DC,1,Rock,1,"Angus Young, Malcolm Young, Brian Johnson",343719,5.73,11170334,10.6529,0.99
2,Balls to the Wall,2,Balls to the Wall,2,Accept,1,Rock,2,"U. Dirkschneider, W. Hoffmann, H. Frank, P. Baltes, S. Kaufmann, G. Hoffmann",342562,5.71,5510424,5.2551,0.99
3,Fast As a Shark,3,Restless and Wild,2,Accept,1,Rock,2,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Hoffman",230619,3.84,3990994,3.8061,0.99
4,Restless and Wild,3,Restless and Wild,2,Accept,1,Rock,2,"F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. Dirkscneider & W. Hoffman",252051,4.20,4331779,4.1311,0.99
5,Princess of the Dawn,3,Restless and Wild,2,Accept,1,Rock,2,Deaffy & R.A. Smith-Diesel,375418,6.26,6290521,5.9991,0.99
6,Put The Finger On You,1,For Those About To Rock We Salute You,1,AC/DC,1,Rock,1,"Angus Young, Malcolm Young, Brian Johnson",205662,3.43,6713451,6.4024,0.99
7,Let's Get It Up,1,For Those About To Rock We Salute You,1,AC/DC,1,Rock,1,"Angus Young, Malcolm Young, Brian Johnson",233926,3.90,7636561,7.2828,0.99
8,Inject The Venom,1,For Those About To Rock We Salute You,1,AC/DC,1,Rock,1,"Angus Young, Malcolm Young, Brian Johnson",210834,3.51,6852860,6.5354,0.99
9,Snowballed,1,For Those About To Rock We Salute You,1,AC/DC,1,Rock,1,"Angus Young, Malcolm Young, Brian Johnson",203102,3.39,6599424,6.2937,0.99
10,Evil Walks,1,For Those About To Rock We Salute You,1,AC/DC,1,Rock,1,"Angus Young, Malcolm Young, Brian Johnson",263497,4.39,8611245,8.2123,0.99


In [0]:
%sql
CREATE OR REPLACE TABLE silver_customer AS
WITH customer_spending AS (
    SELECT
        c.CustomerId,
        SUM(il.UnitPrice * il.Quantity) AS total_spending
    FROM workspace.default.chinook_customer c
    LEFT JOIN workspace.default.chinook_invoice i ON c.CustomerId = i.CustomerId
    LEFT JOIN workspace.default.chinook_invoiceline il ON i.InvoiceId = il.InvoiceId
    GROUP BY c.CustomerId
),
percentile_thresholds AS (
    SELECT
        PERCENTILE_CONT(0.80) WITHIN GROUP (ORDER BY total_spending) AS high_threshold,
        PERCENTILE_CONT(0.60) WITHIN GROUP (ORDER BY total_spending) AS medium_threshold
    FROM customer_spending
)
SELECT
    c.CustomerId,
    TRIM(c.FirstName) AS FirstName,
    TRIM(c.LastName) AS LastName,
    TRIM(c.Company) AS Company,
    TRIM(c.Country) AS Country,
    TRIM(c.Email) AS Email,
    c.SupportRepId,
    CASE
        WHEN COALESCE(cs.total_spending, 0) >= pt.high_threshold THEN 'High'
        WHEN COALESCE(cs.total_spending, 0) >= pt.medium_threshold THEN 'Medium'
        ELSE 'Low'
    END AS SpendingTier
FROM workspace.default.chinook_customer c
LEFT JOIN customer_spending cs ON c.CustomerId = cs.CustomerId
CROSS JOIN percentile_thresholds pt
WHERE c.CustomerId IS NOT NULL
  AND c.Country IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY c.CustomerId ORDER BY c.LastName, c.FirstName) = 1;

SELECT * FROM silver_customer

CustomerId,FirstName,LastName,Company,Country,Email,SupportRepId,SpendingTier
1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,Brazil,luisg@embraer.com.br,3,Medium
2,Leonie,Köhler,null,Germany,leonekohler@surfeu.de,5,Low
3,François,Tremblay,null,Canada,ftremblay@gmail.com,3,Medium
4,Bjørn,Hansen,null,Norway,bjorn.hansen@yahoo.no,4,Medium
5,František,Wichterlová,JetBrains s.r.o.,Czech Republic,frantisekw@jetbrains.com,4,High
6,Helena,Holý,null,Czech Republic,hholy@gmail.com,5,High
7,Astrid,Gruber,null,Austria,astrid.gruber@apple.at,5,High
8,Daan,Peeters,null,Belgium,daan_peeters@apple.be,4,Low
9,Kara,Nielsen,null,Denmark,kara.nielsen@jubii.dk,4,Low
10,Eduardo,Martins,Woodstock Discos,Brazil,eduardo@woodstock.com.br,4,Low


In [0]:
%sql
CREATE OR REPLACE TABLE silver_employee AS
SELECT
    EmployeeId,
    TRIM(LastName) AS LastName,
    TRIM(FirstName) AS FirstName,
    TRIM(Title) AS Title
FROM workspace.default.chinook_employee;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE silver_invoice AS
SELECT
    i.InvoiceId,
    i.CustomerId,
    i.InvoiceDate,
    MONTH(i.InvoiceDate) AS Month,
    QUARTER(i.InvoiceDate) AS Quarter,
    YEAR(i.InvoiceDate) AS Year,
    CONCAT(YEAR(i.InvoiceDate), '-Q', QUARTER(i.InvoiceDate)) AS YearQuarter,
    i.Total
FROM workspace.default.chinook_invoice i
WHERE i.InvoiceId IS NOT NULL
  AND i.CustomerId IS NOT NULL
  AND i.InvoiceDate IS NOT NULL;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE silver_invoiceline AS
SELECT
    il.InvoiceLineId,
    il.InvoiceId,
    c.CustomerId,
    e.EmployeeId,
    il.TrackId,
    il.UnitPrice,
    il.Quantity,
    ROUND(il.UnitPrice * il.Quantity, 2) AS LineAmount
FROM workspace.default.chinook_invoiceline il
LEFT JOIN silver_invoice i
    ON il.InvoiceId = i.InvoiceId
LEFT JOIN silver_customer c
    ON i.CustomerId = c.CustomerId
LEFT JOIN silver_employee e
    ON c.SupportRepId = e.EmployeeId
LEFT JOIN silver_track t
    ON il.TrackId = t.TrackId
WHERE il.InvoiceLineId IS NOT NULL
  AND il.InvoiceId IS NOT NULL
  AND i.CustomerId IS NOT NULL
  AND il.TrackId IS NOT NULL
  AND e.EmployeeId IS NOT NULL
  AND il.Quantity > 0
  AND il.UnitPrice >= 0;

SELECT * FROM silver_invoiceline LIMIT 10;

InvoiceLineId,InvoiceId,CustomerId,EmployeeId,TrackId,UnitPrice,Quantity,LineAmount
1,1,2,5,2,0.99,1,0.99
2,1,2,5,4,0.99,1,0.99
3,2,4,4,6,0.99,1,0.99
4,2,4,4,8,0.99,1,0.99
5,2,4,4,10,0.99,1,0.99
6,2,4,4,12,0.99,1,0.99
7,3,8,4,16,0.99,1,0.99
8,3,8,4,20,0.99,1,0.99
9,3,8,4,24,0.99,1,0.99
10,3,8,4,28,0.99,1,0.99


In [0]:
%sql
-- Data Quality Check: Reconcile invoice totals against line item sums
-- Identifies invoices where stored Total != SUM(line items)

WITH line_item_totals AS (
    SELECT
        InvoiceId,
        ROUND(SUM(UnitPrice * Quantity), 2) AS calculated_total
    FROM silver_invoiceline
    GROUP BY InvoiceId
),
reconciliation AS (
    SELECT
        i.InvoiceId,
        i.Total AS stored_total,
        COALESCE(lit.calculated_total, 0.00) AS calculated_total,
        ROUND(i.Total - COALESCE(lit.calculated_total, 0.00), 2) AS difference
    FROM silver_invoice i
    LEFT JOIN line_item_totals lit
        ON i.InvoiceId = lit.InvoiceId
    WHERE ABS(i.Total - COALESCE(lit.calculated_total, 0.00)) > 0.01
)
SELECT
    COUNT(*) AS mismatched_invoices,
    SUM(ABS(difference)) AS total_discrepancy
FROM reconciliation;

mismatched_invoices,total_discrepancy
0,null


In [0]:
%sql
-- 1. Top Revenue by Genre per Country
-- Which music genres generate the most revenue in each country?


WITH genre_revenue_by_country AS (
    SELECT
        c.Country,
        t.GenreName AS Genre,
        ROUND(SUM(il.UnitPrice * il.Quantity), 2) AS TotalRevenue,
        ROW_NUMBER() OVER (PARTITION BY c.Country ORDER BY SUM(il.UnitPrice * il.Quantity) DESC, t.GenreName ASC) AS rank
    FROM silver_invoiceline il
    INNER JOIN silver_customer c
        ON il.CustomerId = c.CustomerId
    INNER JOIN silver_track t  
        ON il.TrackId = t.TrackId
    GROUP BY
        c.Country,
        t.GenreName
)
SELECT
    Country,
    Genre,
    TotalRevenue,
    rank AS GenreRank
FROM genre_revenue_by_country
WHERE rank = 1
ORDER BY
    Country,
    GenreRank;

Country,Genre,TotalRevenue,GenreRank
Argentina,Alternative & Punk,8.91,1
Australia,Rock,21.78,1
Austria,Rock,14.85,1
Belgium,Rock,20.79,1
Brazil,Rock,80.19,1
Canada,Rock,105.93,1
Chile,Rock,8.91,1
Czech Republic,Rock,24.75,1
Denmark,Rock,20.79,1
Finland,Rock,17.82,1


In [0]:
%sql
-- 2. Customer Segmentation by Spending Tier
-- Using dynamic percentiles instead of fixed thresholds

CREATE OR REPLACE TABLE gold_customer_spending AS WITH customer_spending AS (
    SELECT
        c.CustomerId,
        c.FirstName,
        c.LastName,
        SUM(il.UnitPrice * il.Quantity) AS total_spending
    FROM silver_invoiceline il
    JOIN silver_invoice i ON il.InvoiceId = i.InvoiceId
    JOIN silver_customer c ON i.CustomerId = c.CustomerId
    GROUP BY c.CustomerId, c.FirstName, c.LastName
),
percentile_thresholds AS (
    SELECT
        PERCENTILE_CONT(0.80) WITHIN GROUP (ORDER BY total_spending) AS high_threshold,
        PERCENTILE_CONT(0.60) WITHIN GROUP (ORDER BY total_spending) AS medium_threshold
    FROM customer_spending
),
customer_tiers AS (
    SELECT
        cs.CustomerId,
        cs.FirstName,
        cs.LastName,
        cs.total_spending,
        CASE
            WHEN cs.total_spending >= pt.high_threshold THEN 'High'
            WHEN cs.total_spending >= pt.medium_threshold THEN 'Medium'
            ELSE 'Low'
        END AS spending_tier
    FROM customer_spending cs
    CROSS JOIN percentile_thresholds pt
)
SELECT
    spending_tier,
    COUNT(*) AS customer_count,
    MIN(total_spending) AS min_spending,
    MAX(total_spending) AS max_spending,
    ROUND(AVG(total_spending), 2) AS avg_spending
FROM customer_tiers
GROUP BY spending_tier
ORDER BY
    CASE spending_tier
        WHEN 'High' THEN 1
        WHEN 'Medium' THEN 2
        WHEN 'Low' THEN 3
    END;

SELECT * FROM gold_customer_spending;

spending_tier,customer_count,min_spending,max_spending,avg_spending
High,14,40.62,49.62,43.91
Medium,14,38.62,39.62,39.19
Low,31,36.64,37.62,37.59


In [0]:
%sql
-- 3. Monthly Sales Trends
-- How has revenue trended month-by-month over the last 2 years?

CREATE OR REPLACE TABLE gold_monthly_sales_trends AS 
SELECT
    DATE_FORMAT(i.InvoiceDate, 'yyyy-MM') AS Month,
    ROUND(SUM(il.UnitPrice * il.Quantity), 2) AS Revenue,
    COUNT(DISTINCT i.InvoiceId) AS NumberOfInvoices,
    SUM(il.Quantity) AS TotalUnitsSold
FROM silver_invoice AS i
INNER JOIN silver_invoiceline AS il
    ON i.InvoiceId = il.InvoiceId
WHERE i.InvoiceDate >= ADD_MONTHS(
    (SELECT MAX(InvoiceDate) FROM silver_invoice),
    -23
)
GROUP BY DATE_FORMAT(i.InvoiceDate, 'yyyy-MM')
ORDER BY Month;

SELECT * FROM gold_monthly_sales_trends ORDER BY Month;

Month,Revenue,NumberOfInvoices,TotalUnitsSold
2024-01,22.77,5,23
2024-02,37.62,7,38
2024-03,37.62,7,38
2024-04,37.62,7,38
2024-05,37.62,7,38
2024-06,37.62,7,38
2024-07,39.62,7,38
2024-08,47.62,7,38
2024-09,46.71,6,29
2024-10,42.62,7,38


In [0]:
%sql
SELECT
    CONCAT(e.FirstName, ' ', e.LastName) AS EmployeeName,
    SUM(fact.LineAmount) AS RevenueTotal,
    YEAR(i.InvoiceDate) AS InvoiceYear,
    QUARTER(i.InvoiceDate) AS InvoiceQuarter,
    CONCAT(YEAR(i.InvoiceDate),'-Q',QUARTER(i.InvoiceDate)) AS YearQuarter

FROM silver_invoiceline fact
LEFT JOIN silver_employee e
    ON fact.EmployeeId = e.EmployeeId
LEFT JOIN silver_invoice i
    ON fact.InvoiceId = i.InvoiceId

GROUP BY
    CONCAT(e.FirstName, ' ', e.LastName),
    YEAR(i.InvoiceDate),
    QUARTER(i.InvoiceDate),
    CONCAT(YEAR(i.InvoiceDate),'-Q',QUARTER(i.InvoiceDate))

HAVING RevenueTotal IS NOT NULL
ORDER BY RevenueTotal DESC;

EmployeeName,RevenueTotal,InvoiceYear,InvoiceQuarter,YearQuarter
Jane Peacock,70.45,2022,1,2022-Q1
Jane Peacock,69.30,2022,4,2022-Q4
Margaret Park,66.50,2024,3,2024-Q3
Margaret Park,60.39,2024,1,2024-Q1
Jane Peacock,58.41,2023,3,2023-Q3
Steve Johnson,57.57,2022,1,2022-Q1
Jane Peacock,55.47,2024,4,2024-Q4
Jane Peacock,54.45,2021,3,2021-Q3
Steve Johnson,51.48,2021,2,2021-Q2
Margaret Park,51.48,2021,4,2021-Q4


In [0]:
%sql
-- 5. Popular Tracks by Quantity Sold 
-- What are the top 20 tracks by total units sold, and which albums/artists do they belong to?
CREATE OR REPLACE TABLE gold_popular_tracks AS
SELECT 
    t.TrackName AS track_name,
    t.Title AS album_title,
    t.ArtistName AS artist_name,
    SUM(il.Quantity) AS total_units_sold,
    ROUND(AVG(il.UnitPrice), 2) AS avg_unit_price
FROM silver_invoiceline AS il
INNER JOIN silver_track AS t
    ON il.TrackId = t.TrackId
GROUP BY 
    t.TrackName,
    t.Title,
    t.ArtistName
ORDER BY total_units_sold DESC
LIMIT 20;

SELECT * FROM gold_popular_tracks ORDER BY total_units_sold DESC;

track_name,album_title,artist_name,total_units_sold,avg_unit_price
Give Peace a Chance,Instant Karma: The Amnesty International Campaign to Save Darfur,U2,2,0.99
Nice Guys Finish Last,International Superhits,Green Day,2,0.99
Don't Look Now,"Chronicle, Vol. 2",Creedence Clearwater Revival,2,0.99
For the Greater Good of God,A Matter of Life and Death,Iron Maiden,2,0.99
Menino De Rua,Compositores,O Terço,2,0.99
Release,Tangents,The Tea Party,2,0.99
Shoot Me Again,St. Anger,Metallica,2,0.99
Sunshine Of Your Love,The Cream Of Clapton,Eric Clapton,2,0.99
Lixo Do Mangue,Da Lama Ao Caos,Chico Science & Nação Zumbi,2,0.99
Blood Brothers,Rock In Rio [CD1],Iron Maiden,2,0.99


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- 6. Regional Pricing Insights
-- Do average unit prices differ across countries or regions?
CREATE OR REPLACE TABLE gold_regional_pricing AS
SELECT 
    c.Country AS country,
    COUNT(DISTINCT inv.InvoiceId) AS number_of_invoices,
    SUM(il.Quantity) AS total_units_sold,
    ROUND(AVG(il.UnitPrice), 2) AS avg_unit_price
FROM silver_invoiceline AS il
INNER JOIN silver_invoice AS inv
    ON il.InvoiceId = inv.InvoiceId
INNER JOIN silver_customer AS c
    ON inv.CustomerId = c.CustomerId
GROUP BY c.Country
ORDER BY avg_unit_price DESC;

SELECT * FROM gold_regional_pricing ORDER BY avg_unit_price DESC;

country,number_of_invoices,total_units_sold,avg_unit_price
Chile,7,38,1.23
Hungary,7,38,1.20
Ireland,7,38,1.20
Czech Republic,14,76,1.19
Austria,7,38,1.12
Finland,7,38,1.10
Netherlands,7,38,1.07
USA,91,494,1.06
Norway,7,38,1.04
France,35,190,1.03


Databricks visualization. Run in Databricks to view.